# Stage 1.2 EDA

Exploratory diagnostics for the Stage 1.1 master dataset. This notebook supports Stage 1.3 feature decisions and does not create production features or labels.

## Setup

Load libraries, configure paths, and use one plot style for all sections.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.style.use("seaborn-v0_8")
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "processed" / "master_dataset.parquet"
CALENDAR_DIR = REPO_ROOT / "data" / "calendars"
FIG_DIR = REPO_ROOT / "data" / "reports" / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH).sort_index()
training_df = df.loc["2010-01-01":]

## Section 1 - Master Overview

Question: are the data what we think they are?

In [ ]:
overview = pd.DataFrame({
    "shape_rows": [df.shape[0]],
    "shape_cols": [df.shape[1]],
    "start": [df.index.min()],
    "end": [df.index.max()],
})
trading_days = df.groupby(df.index.year).size().rename("trading_days")
missing = training_df.isna()

fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(missing.T, aspect="auto", interpolation="nearest", cmap="Greys")
ax.set_title("Missing values after 2010-01-01")
ax.set_yticks(range(len(missing.columns)), missing.columns)
ax.set_xlabel("Trading day index")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_missing_values.png", dpi=140)
plt.show()

display(overview)
display(df.dtypes.rename("dtype"))
display(trading_days)
display(pd.concat([df.head(), df.tail()]))

**Comment:** The master dataset has the expected 11 columns, spans 2007-01-03 to 2025-12-30, and has no post-2010 missing values in core columns. Annual trading-day counts are close to the expected ~250.

## Section 2 - US100 Price And Volume

Question: how do price, volume, and daily returns behave across the full sample?

In [ ]:
returns = df["us100_close"].pct_change().dropna()
desc = stats.describe(returns)
top_moves = pd.concat([
    (returns.nlargest(10) * 100).rename("top_positive_pct"),
    (returns.nsmallest(10) * 100).rename("top_negative_pct"),
], axis=1)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2, 1, 1]})
df["us100_close"].plot(ax=axes[0], logy=True, title="US100 close, log scale")
axes[1].bar(df.index, df["us100_volume"], width=1.0)
axes[1].set_title("US100 volume")
axes[1].set_xticks([])
returns.plot(ax=axes[2], kind="hist", bins=90, density=True, alpha=0.55, title="Daily returns")
returns.plot(ax=axes[2], kind="kde", color="black")
for ax in axes:
    ax.set_xlabel("")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_price_volume_returns.png", dpi=140)
plt.show()

display(desc)
display(top_moves.round(2))

**Comment:** Largest moves cluster around GFC, COVID, and 2025 tariff shock dates, which looks like real market stress rather than obvious bad ticks. Returns are fat-tailed, with kurtosis around 8.

## Section 3 - Distribution Shift Across Regimes

Question: does 2010 behave like 2024?

In [ ]:
windows = {
    "2010-2012": ("2010-01-01", "2012-12-31"),
    "2013-2015": ("2013-01-01", "2015-12-31"),
    "2016-2018": ("2016-01-01", "2018-12-31"),
    "2019-2021": ("2019-01-01", "2021-12-31"),
    "2022-2024": ("2022-01-01", "2024-12-31"),
}
rows = []
for label, (start, end) in windows.items():
    w = df.loc[start:end]
    r = w["us100_close"].pct_change().dropna()
    intraday_range = ((w["us100_high"] - w["us100_low"]) / w["us100_close"]).dropna()
    rows.append({"window": label, "return_mean_pct": r.mean() * 100, "return_std_pct": r.std() * 100,
                 "vix_mean": w["vix_close"].mean(), "vix_std": w["vix_close"].std(),
                 "volume_mean_bn": w["us100_volume"].mean() / 1e9, "volume_std_bn": w["us100_volume"].std() / 1e9,
                 "range_mean_pct": intraday_range.mean() * 100, "range_std_pct": intraday_range.std() * 100})
regime = pd.DataFrame(rows).set_index("window")

plot_cols = ["return_std_pct", "vix_mean", "volume_mean_bn", "range_mean_pct"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), plot_cols):
    regime[col].plot(ax=ax, kind="bar", title=col)
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_distribution_shift.png", dpi=140)
plt.show()
display(regime.round(3))

**Comment:** 2019-2021 and 2022-2024 are materially higher-volatility and higher-volume regimes than 2013-2018. Stage 1.6 should treat time decay as a serious tuning dimension.

## Section 4 - Correlations

Question: do SPX, DXY, and VIX add non-redundant context relative to US100?

In [ ]:
asset_returns = pd.DataFrame({
    "us100": df["us100_close"].pct_change(),
    "vix": df["vix_close"].pct_change(),
    "spx": df["spx_close"].pct_change(),
    "dxy": df["dxy_close"].pct_change(),
}).dropna()
corr = asset_returns.corr()
rolling = pd.DataFrame({
    "us100_spx": asset_returns["us100"].rolling(60).corr(asset_returns["spx"]),
    "us100_dxy": asset_returns["us100"].rolling(60).corr(asset_returns["dxy"]),
    "us100_vix": asset_returns["us100"].rolling(60).corr(asset_returns["vix"]),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
axes[0].set_xticks(range(len(corr)), corr.columns)
axes[0].set_yticks(range(len(corr)), corr.index)
axes[0].set_title("Pearson correlation on daily returns")
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[0].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=axes[0], fraction=0.046)
rolling.plot(ax=axes[1], title="Rolling 60-day correlations")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_correlations.png", dpi=140)
plt.show()
display(corr.round(3))

**Comment:** SPX is highly correlated with US100 but not perfectly redundant. VIX adds a strong inverse risk-regime signal; DXY is weaker and more episodic.

## Section 5 - Volatility And ATR-like Statistics

Question: how large are typical moves? This section intentionally avoids classic ATR; ATR is only used in horizon calibration.

In [ ]:
rolling_std = returns.rolling(14).std() * 100
intraday_range = ((df["us100_high"] - df["us100_low"]) / df["us100_close"]).dropna() * 100
rolling_range = intraday_range.rolling(14).mean()
range_stats = intraday_range.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

fig, axes = plt.subplots(3, 1, figsize=(12, 9))
rolling_std.plot(ax=axes[0], title="Rolling 14-day return std (%)")
rolling_range.plot(ax=axes[1], title="Rolling 14-day mean intraday range (%)")
intraday_range.plot(ax=axes[2], kind="hist", bins=80, title="Intraday range distribution (%)")
for ax in axes:
    ax.set_xlabel("")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_volatility_proxy.png", dpi=140)
plt.show()
display(range_stats.round(3))

**Comment:** Mean intraday range is about 1.49%, with a 90th percentile near 2.66%. That gives practical context for why ATR-based barriers may often resolve within a few weeks.

## Section 6 - Calendars Sanity Check

Question: are FOMC, CPI, and NFP calendars internally plausible?

In [ ]:
calendar_specs = {
    "fomc": (CALENDAR_DIR / "fomc_dates.parquet", "meeting_date"),
    "cpi": (CALENDAR_DIR / "cpi_release_dates.parquet", "release_date"),
    "nfp": (CALENDAR_DIR / "nfp_release_dates.parquet", "release_date"),
}
calendar_counts = {}
calendar_dates = []
for name, (path, date_col) in calendar_specs.items():
    cal = pd.read_parquet(path)
    dates = pd.to_datetime(cal[date_col])
    calendar_counts[name] = dates.dt.year.value_counts().sort_index()
    calendar_dates.append(pd.DataFrame({"calendar": name, "date": dates}))
counts = pd.DataFrame(calendar_counts).fillna(0).astype(int)
events = pd.concat(calendar_dates, ignore_index=True)
recent = events[events["date"] >= (df.index.max() - pd.DateOffset(years=3))]

fig, ax = plt.subplots(figsize=(12, 4))
for y, (name, group) in enumerate(recent.groupby("calendar")):
    ax.scatter(group["date"], [y] * len(group), label=name, s=18)
ax.set_yticks(range(recent["calendar"].nunique()), sorted(recent["calendar"].unique()))
ax.set_title("Calendar event dates, recent 3 years")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_calendar_events.png", dpi=140)
plt.show()
display(counts)

**Comment:** Completed years show the expected cadence: FOMC around 8 meetings per year, CPI and NFP around 12 releases per year. Partial boundary years explain 2007 and 2026 counts in CPI/NFP files.

## Summary Findings

- **Master dataset shape is as expected:** 4779 rows, 11 columns, and a 2007-01-03 to 2025-12-30 range. Implication for 1.3: proceed with feature design on the Stage 1.1 master file.
- **No post-2010 core missingness:** training-period OHLCV and context columns have zero NaNs. Implication for 1.3: no special missing-value policy is needed for core inputs yet.
- **Return distribution is fat-tailed:** daily US100 returns have kurtosis around 8. Implication for 1.6: robust validation across stress windows matters more than average-period performance.
- **Largest return outliers align with known stress events:** GFC, COVID, and 2025 tariff-shock dates dominate the extremes. Implication for 1.3: keep outliers unless a later audit identifies source errors.
- **Regime shift is material:** 2019-2021 and 2022-2024 show much higher volatility and volume than 2013-2018. Implication for 1.6: tune time decay and do not assume stationarity.
- **SPX is correlated but not identical:** US100-SPX daily return correlation is about 0.93. Implication for 1.3: SPX context can remain, but marginal value should be checked during feature selection.
- **VIX adds risk-regime information:** US100-VIX return correlation is about -0.70. Implication for 1.3: VIX-derived context is likely worth testing.
- **DXY relation is weaker:** US100-DXY return correlation is about -0.14. Implication for 1.3: DXY features should be treated as optional context, not a core driver.
- **Typical intraday range is meaningful:** mean range is about 1.49%, with the 90th percentile near 2.66%. Implication for calibration: ATR-based barriers should often resolve within weeks rather than months.
- **Calendars pass cadence sanity:** completed years mostly show expected FOMC/CPI/NFP counts. Implication for 1.3: event-distance features can use these calendars, subject to normal as-of handling.